# 한국 도로 외부 데이터: YOLOv8s와 RT-DETR 비교

**목적:** 기존 7주차 체크포인트와 threshold를 그대로 새 한국 도로 이미지에 적용하여 성능 수준과 모델 간 차이가 유지되는지 확인합니다. 추가 학습과 새 test에서의 threshold 선택은 없습니다.

**입력:** 같은 폴더에 `20210819_172456_맑음_오후_510.jpg`(또는 PNG)와 `20210819_172456_맑음_오후_510.json`. 하위 폴더도 검색하지만 각 쌍은 같은 폴더에 있어야 합니다. 원본 파일을 변경하거나 복사하지 않습니다.

**실행 순서:** 셀 1~5 → JSON 좌표 형식을 확인하고 셀 6 수정 → 셀 7 정답 박스 점검 → 셀 8 모델 준비 → 셀 9의 두 확인값 설정 → 셀 10 결과 확인.

**아직 실제 JSON 파일을 제공받지 않았습니다.** `annotations`, `images`, `categories`를 사용하는 AI Hub 형태를 지원합니다. 알 수 없는 구조, 숫자 클래스의 의미, bbox 순서는 추정하지 않습니다. 셀 5의 출력 또는 `json_schema_samples.json`과 원 JSON 한 개를 공유하면 추가 구조에 맞출 수 있습니다.

7주차 완료 폴더의 `protocol.json`, `decisions_locked.json`, `completed.json`과 원 가중치가 필요합니다. 가중치 경로가 바뀌면 셀 4의 `path_remap` 또는 `weight_overrides`를 설정하세요. 이전 data.yaml과 임시 staging 폴더는 필요 없습니다.


## 셀 1 — 설치
Colab에서 **런타임 → 런타임 유형 변경 → T4 GPU**를 권장합니다. 설치 뒤 이미 ultralytics를 import한 런타임이면 세션을 재시작하고 셀 2부터 실행하세요. 기존 CUDA torch를 CPU 전용으로 재설치하지 않습니다.


In [ ]:
%pip install -q "ultralytics==8.4.183" pycocotools pandas matplotlib Pillow pyyaml


## 셀 2 — Google Drive 연결


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image as DisplayImage


## 셀 3 — 평가 엔진 불러오기
필요한 코드가 이 노트북에 포함되어 있습니다. 별도 `.py` 업로드는 필요 없습니다. AP 평가기의 간단한 정상/빈 예측 검사도 수행합니다.


In [ ]:
#@title 평가 엔진 설치 및 AP 검사 (그대로 실행)
from pathlib import Path
import importlib.util
import sys
ENGINE_SOURCE = '"""Frozen-checkpoint external evaluation; no training or test-set tuning.\n\nOne same-stem JSON beside each image. Unknown label/coordinate schemas fail closed.\nOperating-point matching follows week7: score-ordered best unmatched GT, IoU=.5.\n"""\nfrom pathlib import Path\nfrom collections import Counter, defaultdict\nfrom datetime import datetime, timezone\nimport contextlib\nimport gc\nimport hashlib\nimport io\nimport json\nimport math\nimport shutil\nimport unicodedata\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image, ImageDraw\n\nVERSION = \'1.0.0\'\nEXTS = {\'.jpg\', \'.jpeg\', \'.png\', \'.bmp\', \'.webp\', \'.tif\', \'.tiff\'}\nMODELS = (\'yolo\', \'rtdetr\')\n\n\ndef require(ok, message):\n    if not ok:\n        raise ValueError(message)\n\n\ndef clean(x):\n    if isinstance(x, dict):\n        return {str(k): clean(v) for k, v in x.items()}\n    if isinstance(x, (list, tuple)):\n        return [clean(v) for v in x]\n    if isinstance(x, np.generic):\n        return clean(x.item())\n    if isinstance(x, Path):\n        return str(x)\n    if isinstance(x, float) and not math.isfinite(x):\n        return None\n    return x\n\n\ndef save_json(path, data):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + \'.tmp\')\n    tmp.write_text(json.dumps(clean(data), ensure_ascii=False, indent=2, allow_nan=False), encoding=\'utf-8\')\n    tmp.replace(path)\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding=\'utf-8-sig\'))\n\n\ndef digest(obj):\n    # Same canonical serialization as week7; do not change separators.\n    return hashlib.sha256(json.dumps(obj, sort_keys=True, ensure_ascii=False, allow_nan=False).encode()).hexdigest()\n\n\ndef file_sha(path):\n    h = hashlib.sha256()\n    with open(path, \'rb\') as f:\n        for block in iter(lambda: f.read(2**20), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef norm(s):\n    return unicodedata.normalize(\'NFC\', str(s)).strip().casefold()\n\n\ndef ref_stem(s):\n    return norm(Path(str(s).replace(\'\\\\\', \'/\')).stem)\n\n\ndef rgb_image(path):\n    with Image.open(path) as im:\n        require(im.getexif().get(274, 1) == 1,\n                f\'EXIF 회전이 있는 이미지입니다. JSON 좌표 방향 확인 후 별도 변환 필요: {path}\')\n        if \'A\' in im.getbands():\n            require(im.getchannel(\'A\').getextrema() == (255, 255), f\'투명 픽셀 처리 기준을 정해야 합니다: {path}\')\n        return np.asarray(im.convert(\'RGB\')).copy()\n\n\ndef pixel_sha(rgb):\n    return hashlib.sha256(str(rgb.shape).encode() + rgb.tobytes()).hexdigest()\n\n\ndef inventory(cfg):\n    root = Path(cfg[\'data_dir\']).expanduser().resolve()\n    require(root.is_dir(), f\'이미지와 JSON 폴더가 없습니다: {root}\')\n    paths = sorted(root.rglob(\'*\') if cfg.get(\'recursive\', True) else root.iterdir())\n    images, labels = defaultdict(list), defaultdict(list)\n    for p in paths:\n        if not p.is_file():\n            continue\n        key = (str(p.parent), norm(p.stem))\n        if p.suffix.lower() in EXTS:\n            images[key].append(p)\n        elif p.suffix.lower() == \'.json\':\n            labels[key].append(p)\n    require(images, f\'이미지가 없습니다: {root}\')\n    ambiguous = [str(k) for k, v in images.items() if len(v) != 1]\n    ambiguous += [str(k) for k, v in labels.items() if k in images and len(v) != 1]\n    require(not ambiguous, \'동일 이름 파일이 중복됩니다:\\n\' + \'\\n\'.join(ambiguous[:10]))\n    rows = []\n    for key, ps in images.items():\n        p = ps[0]\n        js = labels.get(key, [])\n        rows.append({\'image\': str(p), \'json\': str(js[0]) if len(js) == 1 else \'\',\n                     \'relative_image\': str(p.relative_to(root)), \'status\': \'paired\' if len(js) == 1 else \'MISSING_JSON\'})\n    frame = pd.DataFrame(rows)\n    orphan = [str(p) for k, ps in labels.items() if k not in images for p in ps]\n    print(f\'이미지 {len(frame)}장 / JSON 연결 {(frame.status == "paired").sum()}장 / 누락 {(frame.status != "paired").sum()}장\')\n    print(f\'이미지와 연결되지 않은 JSON {len(orphan)}개 (메타데이터일 수도 있음)\')\n    return frame, orphan\n\n\ndef annotation_list(obj):\n    require(isinstance(obj, dict), \'JSON 최상위가 dict가 아닙니다. 샘플 JSON으로 어댑터 추가 필요\')\n    require(\'annotations\' in obj, f\'annotations 필드가 없습니다. 현재 키: {list(obj)}. 누락 라벨을 빈 정답으로 처리하지 않습니다.\')\n    ann = obj[\'annotations\']\n    if isinstance(ann, dict):\n        # Single annotation object only; id-keyed maps are intentionally not guessed.\n        require(\'bbox\' in ann or \'label\' in ann or \'category_id\' in ann, \'알 수 없는 annotations dict 구조\')\n        ann = [ann]\n    require(isinstance(ann, list) and all(isinstance(a, dict) for a in ann), \'annotations는 객체 dict의 list여야 합니다.\')\n    return ann\n\n\ndef image_meta(obj, image_path, w, h):\n    records = obj.get(\'images\', obj.get(\'image\'))\n    if records is None:\n        return None\n    records = [records] if isinstance(records, dict) else records\n    require(isinstance(records, list) and all(isinstance(r, dict) for r in records), \'images 메타데이터 구조 확인 필요\')\n    if len(records) == 1:\n        rec = records[0]\n    else:\n        found = [r for r in records if ref_stem(r.get(\'file_name\', r.get(\'filename\', \'\'))) == norm(Path(image_path).stem)]\n        require(len(found) == 1, \'JSON 내부의 이미지 참조를 하나로 연결할 수 없습니다.\')\n        rec = found[0]\n    name = rec.get(\'file_name\', rec.get(\'filename\'))\n    if name:\n        require(ref_stem(name) == norm(Path(image_path).stem), f\'파일명과 JSON 내부 file_name 불일치: {name}\')\n    for key, actual in [(\'width\', w), (\'height\', h)]:\n        if key in rec:\n            require(float(rec[key]) == actual, f\'JSON {key}={rec[key]}, 실제={actual}; 리사이즈된 원본인지 확인하세요.\')\n    return rec.get(\'id\')\n\n\ndef category_map(obj, cfg):\n    result = {str(k): str(v) for k, v in cfg.get(\'category_id_to_name\', {}).items()}\n    cats = obj.get(\'categories\', [])\n    cats = [cats] if isinstance(cats, dict) and \'id\' in cats else cats\n    require(isinstance(cats, list), \'categories 형식을 확인하세요. list of {id,name}을 지원합니다.\')\n    for c in cats:\n        require(isinstance(c, dict) and \'id\' in c and \'name\' in c, \'categories에 id/name이 필요합니다.\')\n        k, v = str(c[\'id\']), str(c[\'name\'])\n        require(k not in result or norm(result[k]) == norm(v), f\'클래스 매핑 충돌: {k}\')\n        result[k] = v\n    return result\n\n\ndef label_name(a, cats):\n    names = []\n    if \'category_id\' in a:\n        key = str(a[\'category_id\'])\n        require(key in cats, f\'category_id={key}의 이름이 없습니다. category_id_to_name을 실제 클래스표로 설정하세요.\')\n        names.append(cats[key])\n    for key in (\'label\', \'category_name\', \'class_name\'):\n        if key in a:\n            v = a[key]\n            if isinstance(v, dict):\n                require(\'name\' in v, f\'{key} dict의 name 필드가 없습니다: {v}\')\n                v = v[\'name\']\n            require(isinstance(v, (str, int)), f\'{key} 형식 확인 필요: {v}\')\n            value = str(v)\n            if value in cats:\n                value = cats[value]\n            require(not value.isdigit(), f\'숫자 클래스 {value}의 뜻을 확인해 category_id_to_name에 입력하세요.\')\n            names.append(value)\n    require(names, f\'객체의 클래스 이름을 찾지 못했습니다. 객체 키: {list(a)}\')\n    require(len({norm(x) for x in names}) == 1, f\'한 객체에서 클래스 이름 충돌: {names}\')\n    return names[0]\n\n\ndef bbox_xyxy(raw, cfg, w, h):\n    fmt = cfg.get(\'bbox_format\')\n    if isinstance(raw, dict):\n        keys = set(raw)\n        if {\'xmin\', \'ymin\', \'xmax\', \'ymax\'} <= keys:\n            v, fmt = [raw[k] for k in (\'xmin\', \'ymin\', \'xmax\', \'ymax\')], \'xyxy\'\n        elif {\'x1\', \'y1\', \'x2\', \'y2\'} <= keys:\n            v, fmt = [raw[k] for k in (\'x1\', \'y1\', \'x2\', \'y2\')], \'xyxy\'\n        elif {\'x\', \'y\', \'width\', \'height\'} <= keys:\n            v, fmt = [raw[k] for k in (\'x\', \'y\', \'width\', \'height\')], \'xywh\'\n        elif {\'x\', \'y\', \'w\', \'h\'} <= keys:\n            v, fmt = [raw[k] for k in (\'x\', \'y\', \'w\', \'h\')], \'xywh\'\n        else:\n            raise ValueError(f\'지원하지 않는 bbox dict: {raw}\')\n    else:\n        v = np.asarray(raw)\n        if v.shape == (2, 2):\n            v, fmt = v.reshape(4), \'xyxy\'\n        else:\n            require(v.shape == (4,), f\'bbox는 숫자 4개 또는 두 모서리여야 합니다: {raw}\')\n            require(fmt in (\'xyxy\', \'xywh\', \'cxcywh\'),\n                    \'숫자 4개의 bbox는 좌표 순서를 자동 확정할 수 없습니다. 미리보기 후 CFG["bbox_format"]을 xyxy 또는 xywh 등으로 설정하세요.\')\n    v = np.asarray(v, dtype=float)\n    require(np.isfinite(v).all(), f\'유한하지 않은 GT 좌표: {raw}\')\n    units = cfg.get(\'bbox_units\', \'pixels\')\n    require(units in (\'pixels\', \'normalized\'), \'bbox_units는 pixels 또는 normalized\')\n    if units == \'normalized\':\n        v = v * np.array([w, h, w, h])\n    else:\n        require(not (np.all(v >= 0) and np.all(v <= 1)), \'모든 bbox 값이 0~1입니다. 정규화 좌표 여부를 확인하세요.\')\n    x, y, a, b = v\n    if fmt == \'xyxy\':\n        box = np.array([x, y, a, b])\n    elif fmt == \'xywh\':\n        box = np.array([x, y, x+a, y+b])\n    else:\n        box = np.array([x-a/2, y-b/2, x+a/2, y+b/2])\n    require(np.all(box[2:] > box[:2]), f\'면적이 0 또는 음수인 GT: {raw} / format={fmt}\')\n    require(np.all(box >= 0) and box[0] < w and box[1] < h and box[2] <= w and box[3] <= h,\n            f\'이미지 밖 GT: {box.tolist()} / image={w}x{h}. format/units 확인. 임의 clipping하지 않습니다.\')\n    return box.tolist()\n\n\ndef parse_label(obj, image_path, w, h, cfg):\n    ann = annotation_list(obj)\n    image_id = image_meta(obj, image_path, w, h)\n    cats = category_map(obj, cfg)\n    wanted = {norm(v) for v in cfg[\'target_names\']}\n    gt, observed, ignored = [], Counter(), 0\n    for a in ann:\n        if \'image_id\' in a:\n            require(image_id is not None and str(a[\'image_id\']) == str(image_id),\n                    \'annotation.image_id와 image.id를 확인하세요. 다른 이미지의 주석을 섞지 않습니다.\')\n        name = label_name(a, cats)\n        observed[name] += 1\n        if norm(name) not in wanted:\n            ignored += 1\n            continue\n        require(not a.get(\'iscrowd\', 0) and not a.get(\'ignore\', 0),\n                \'포트홀 crowd/ignore 영역이 있습니다. 이 평가기는 모든 포트홀을 개별 객체로 평가합니다.\')\n        require(\'bbox\' in a, f\'포트홀 객체에 bbox가 없습니다: {a}\')\n        gt.append(bbox_xyxy(a[\'bbox\'], cfg, w, h))\n    require(len({tuple(b) for b in gt}) == len(gt), \'같은 좌표의 포트홀 GT가 중복됩니다. 원 라벨 확인 필요\')\n    return gt, dict(observed), ignored\n\n\ndef inspect_samples(pairs, cfg, out):\n    out = Path(out)\n    out.mkdir(parents=True, exist_ok=True)\n    pairs.to_csv(out/\'image_json_pairs.csv\', index=False, encoding=\'utf-8-sig\')\n    require((pairs.status == \'paired\').all(), f\'JSON 누락 이미지가 있습니다. {out}/image_json_pairs.csv 확인\')\n    samples = []\n    for r in pairs.head(3).to_dict(\'records\'):\n        obj = read_json(r[\'json\'])\n        samples.append({\'image\': r[\'relative_image\'], \'json\': r[\'json\'], \'top_level_keys\': list(obj) if isinstance(obj, dict) else type(obj).__name__,\n                        \'images\': obj.get(\'images\', obj.get(\'image\')) if isinstance(obj, dict) else None,\n                        \'categories\': obj.get(\'categories\') if isinstance(obj, dict) else None,\n                        \'annotations_sample\': obj.get(\'annotations\', [])[:2] if isinstance(obj, dict) and isinstance(obj.get(\'annotations\'), list) else obj.get(\'annotations\') if isinstance(obj, dict) else obj})\n    save_json(out/\'json_schema_samples.json\', samples)\n    print(json.dumps(samples, ensure_ascii=False, indent=2)[:14000])\n    return samples\n\n\ndef format_preview(pairs, cfg, out):\n    import matplotlib.pyplot as plt\n    rows = pairs.head(3).to_dict(\'records\')\n    fig, axs = plt.subplots(len(rows), 2, figsize=(14, 4*len(rows)), squeeze=False)\n    for i, row in enumerate(rows):\n        rgb = rgb_image(row[\'image\'])\n        h, w = rgb.shape[:2]\n        obj = read_json(row[\'json\'])\n        for j, fmt in enumerate((\'xyxy\', \'xywh\')):\n            ax = axs[i, j]\n            ax.imshow(rgb)\n            ax.axis(\'off\')\n            try:\n                gt, names, _ = parse_label(obj, row[\'image\'], w, h, {**cfg, \'bbox_format\': fmt})\n                for x1, y1, x2, y2 in gt:\n                    ax.add_patch(plt.Rectangle((x1,y1),x2-x1,y2-y1,fill=False,color=\'lime\',lw=2))\n                ax.set_title(f\'{i+1}: {fmt} | target boxes: {len(gt)}\')\n            except Exception as e:\n                ax.set_title(f\'{i+1}: {fmt} | CHECK SCHEMA\', color=\'red\')\n                print(f\'미리보기 {i+1}, {fmt}: {e}\')\n    fig.suptitle(\'Coordinate format candidates - verify against the original JSON specification\')\n    fig.tight_layout()\n    fig.savefig(Path(out)/\'bbox_format_candidates.png\', dpi=140)\n    plt.show()\n\n\ndef prepare(cfg, out):\n    pairs, orphan = inventory(cfg)\n    out = Path(out)\n    pairs.to_csv(out/\'image_json_pairs.csv\', index=False, encoding=\'utf-8-sig\')\n    require((pairs.status == \'paired\').all(), \'누락 JSON을 해결한 뒤 다시 실행하세요.\')\n    rows, errors, seen, labels = [], [], {}, Counter()\n    for n, r in enumerate(pairs.to_dict(\'records\'), 1):\n        try:\n            rgb = rgb_image(r[\'image\'])\n            h, w = rgb.shape[:2]\n            obj = read_json(r[\'json\'])\n            gt, observed, ignored = parse_label(obj, r[\'image\'], w, h, cfg)\n            ident = pixel_sha(rgb)\n            require(ident not in seen, f\'동일 pixel 중복 이미지: {seen.get(ident)}\')\n            seen[ident] = r[\'image\']\n            labels.update(observed)\n            rows.append({**r, \'image_id\': ident, \'width\': w, \'height\': h, \'gt\': gt,\n                         \'image_sha256\': file_sha(r[\'image\']), \'label_sha256\': file_sha(r[\'json\']),\n                         \'other_objects\': ignored,\n                         \'group_hint_unverified\': Path(r[\'image\']).stem.rsplit(\'_\', 1)[0]})\n        except Exception as e:\n            errors.append({\'image\': r[\'image\'], \'json\': r[\'json\'], \'error\': str(e)})\n        if n % 100 == 0 or n == len(pairs):\n            print(f\'라벨 검사: {n}/{len(pairs)}\')\n    pd.DataFrame(errors, columns=[\'image\', \'json\', \'error\']).to_csv(out/\'label_errors.csv\', index=False, encoding=\'utf-8-sig\')\n    save_json(out/\'observed_classes.json\', dict(labels))\n    if errors:\n        print(pd.DataFrame(errors).head(10).to_string(index=False))\n        raise ValueError(f\'라벨 오류 {len(errors)}개. 잘못된 이미지를 제외하고 진행하지 않습니다. label_errors.csv 확인\')\n    objects = sum(len(r[\'gt\']) for r in rows)\n    require(objects > 0, f\'포트홀 GT가 0개입니다. target_names 확인. 관찰 클래스: {dict(labels)}\')\n    small = sum(sum((b[2]-b[0])*(b[3]-b[1]) < 32**2 for b in r[\'gt\']) for r in rows)\n    audit = {\'images\': len(rows), \'objects\': objects, \'small_objects_original_pixels\': small,\n             \'positive_images\': sum(bool(r[\'gt\']) for r in rows), \'negative_images\': sum(not r[\'gt\'] for r in rows),\n             \'observed_classes\': dict(labels), \'orphan_json\': orphan, \'capture_groups_verified\': False,\n             \'coordinate_format\': cfg.get(\'bbox_format\'), \'coordinate_units\': cfg.get(\'bbox_units\', \'pixels\')}\n    save_json(out/\'dataset_audit.json\', audit)\n    save_json(out/\'external_manifest.json\', rows)\n    flat = [{k:v for k,v in r.items() if k != \'gt\'} | {\'gt_count\':len(r[\'gt\'])} for r in rows]\n    pd.DataFrame(flat).to_csv(out/\'dataset_images.csv\', index=False, encoding=\'utf-8-sig\')\n    print(json.dumps(audit, ensure_ascii=False, indent=2))\n    if not audit[\'negative_images\']:\n        print(\'포트홀이 없는 이미지가 0장입니다. FP/image는 이 포트홀 중심 세트의 결과이며 일반 도로 오탐률을 대표하지 않습니다.\')\n    print(\'파일명 group_hint는 확인 전 참고값입니다. 인접 프레임 중복/촬영 구간은 직접 점검하세요.\')\n    return {\'rows\': rows, \'audit\': audit, \'signature\': digest({\'rows\':rows, \'config\':dataset_settings(cfg)})}\n\n\ndef dataset_settings(cfg):\n    return {k:cfg.get(k) for k in (\'data_dir\',\'recursive\',\'target_names\',\'category_id_to_name\',\'bbox_format\',\'bbox_units\')}\n\n\ndef preview_ground_truth(prepared, out):\n    import matplotlib.pyplot as plt\n    rows = prepared[\'rows\']\n    indices = np.linspace(0, len(rows)-1, min(12,len(rows)), dtype=int)\n    fig, axs = plt.subplots(math.ceil(len(indices)/3), 3, figsize=(15,4*math.ceil(len(indices)/3)), squeeze=False)\n    for ax in axs.ravel(): ax.axis(\'off\')\n    for ax, index in zip(axs.ravel(), indices):\n        r = rows[index]\n        ax.imshow(rgb_image(r[\'image\']))\n        for x1,y1,x2,y2 in r[\'gt\']:\n            ax.add_patch(plt.Rectangle((x1,y1),x2-x1,y2-y1,fill=False,color=\'lime\',lw=2))\n        ax.set_title(f\'Image {index+1} | GT={len(r["gt"])}\')\n    fig.tight_layout()\n    fig.savefig(Path(out)/\'ground_truth_preview.png\', dpi=140)\n    plt.show()\n\n\ndef remap(value, cfg):\n    text = str(value)\n    for old, new in sorted(cfg.get(\'path_remap\', {}).items(), key=lambda kv: -len(kv[0])):\n        old = old.rstrip(\'/\')\n        if text == old or text.startswith(old + \'/\'):\n            text = str(new).rstrip(\'/\') + text[len(old):]\n            break\n    return Path(text).expanduser().resolve()\n\n\ndef load_week7(cfg):\n    root = remap(cfg[\'week7_out\'], cfg)\n    p, l, c = [root/name for name in (\'protocol.json\',\'decisions_locked.json\',\'completed.json\')]\n    for f in (p,l,c): require(f.is_file(), f\'7주차 완료 폴더의 파일이 없습니다: {f}\')\n    protocol, lock, done = map(read_json, (p,l,c))\n    require(lock.get(\'protocol_sha256\') == digest(protocol), \'7주차 protocol/lock 해시 불일치\')\n    require(lock.get(\'lock_sha256\') == digest({k:v for k,v in lock.items() if k != \'lock_sha256\'}), \'7주차 lock 해시 불일치\')\n    require(done.get(\'status\') == \'complete\' and done.get(\'lock_sha256\') == lock[\'lock_sha256\'], \'7주차 실행 완료 상태 확인 필요\')\n    size = int(cfg[\'imgsz\'])\n    require(protocol[\'config\'].get(\'match_iou\', .5) == .5, \'이 노트북은 7주차 IoU=0.5 비교용입니다.\')\n    settings = protocol.get(\'inference_by_resolution\', {}).get(str(size))\n    require(settings is not None, f\'7주차에 imgsz={size}의 추론 설정이 없습니다.\')\n    settings = dict(settings)\n    require(settings.get(\'quantize\') == 32 and settings.get(\'batch\') == 1 and not settings.get(\'augment\'), \'7주차 FP32/batch1/noTTA 설정 확인 필요\')\n    records = []\n    thresholds = pd.DataFrame(lock[\'thresholds\'])\n    expected_folds = set(map(int, cfg.get(\'folds\', [1,2,3,4,5])))\n    present = {int(f[\'fold\']) for f in protocol[\'folds\']}\n    require(expected_folds <= present, f\'요청 fold {expected_folds} / 기록 fold {present}\')\n    for fold in protocol[\'folds\']:\n        k = int(fold[\'fold\'])\n        if k not in expected_folds: continue\n        for model in MODELS:\n            override = cfg.get(\'weight_overrides\', {}).get(k, cfg.get(\'weight_overrides\', {}).get(str(k), {})).get(model)\n            weights = remap(override or fold[f\'{model}_weights\'], cfg)\n            require(weights.is_file(), f\'가중치가 없습니다: {weights}. path_remap 또는 weight_overrides를 설정하세요.\')\n            sha = file_sha(weights)\n            require(protocol.get(\'weights_sha256\', {}).get(f\'fold{k}/{model}\') == sha,\n                    f\'7주차와 가중치 해시가 다릅니다: fold{k}/{model}. 다른 가중치에 기존 threshold를 적용하지 않습니다.\')\n            for target in cfg[\'targets\']:\n                select = thresholds[(thresholds.fold == k) & (thresholds.model == model) &\n                                    (thresholds.imgsz == size) & np.isclose(thresholds.target_fp_per_image, target)]\n                require(len(select) == 1, f\'기존 threshold가 하나가 아닙니다: fold{k}/{model}/{size}/{target}\')\n                s = select.iloc[0]\n                require(s[\'split\'] == \'inner_val\', \'threshold 출처는 inner_val이어야 합니다.\')\n                t = float(s.threshold)\n                require(math.isfinite(t) and float(settings[\'conf\']) <= t <= float(np.nextafter(1., np.inf)), \'유효하지 않은 기존 threshold\')\n                records.append({\'fold\':k, \'model\':model, \'weights\':str(weights), \'weight_sha256\':sha,\n                                \'target_fp_per_image\':float(target), \'threshold\':t, \'imgsz\':size})\n    for model in MODELS:\n        subset = {r[\'fold\']:r[\'weight_sha256\'] for r in records if r[\'model\'] == model}\n        require(len(set(subset.values())) == len(subset), f\'{model}: 여러 fold에 동일 가중치가 연결됐습니다.\')\n    sources=[p,l,c]\n    old_table=root/\'outer_grid_by_fold.csv\'\n    if old_table.is_file(): sources.append(old_table)\n    return {\'records\':records, \'inference\':settings, \'protocol\':protocol, \'lock\':lock, \'root\':str(root),\n            \'source_hashes\':{str(f):file_sha(f) for f in sources}}\n\n\ndef validate_prediction(pred):\n    b = np.asarray(pred[\'boxes\'], dtype=float).reshape(-1,4)\n    s = np.asarray(pred[\'scores\'], dtype=float)\n    require(s.ndim == 1 and len(s) == len(b), \'예측 bbox/score 길이 불일치\')\n    require(np.isfinite(b).all() and np.isfinite(s).all(), \'NaN/Inf 예측\')\n    require(np.all((s >= 0) & (s <= 1)), \'범위를 벗어난 confidence\')\n    require(not np.any(b[:,2:] < b[:,:2]), \'역전된 예측 bbox\')\n    return b, s  # finite zero-area predictions retained as FP\n\n\ndef iou_one(box, gt):\n    inter = np.maximum(0, np.minimum(box[2:],gt[:,2:]) - np.maximum(box[:2],gt[:,:2])).prod(axis=1)\n    area = np.maximum(0,box[2:]-box[:2]).prod()\n    areas = np.maximum(0,gt[:,2:]-gt[:,:2]).prod(axis=1)\n    return inter / np.maximum(area+areas-inter, 1e-15)\n\n\ndef match(gt, boxes, scores, threshold):\n    gt = np.asarray(gt, dtype=float).reshape(-1,4)\n    used = np.zeros(len(gt), dtype=bool)\n    assignment = np.full(len(boxes), -1, dtype=int)\n    for j in np.argsort(-scores, kind=\'stable\'):\n        if scores[j] < threshold: continue\n        if len(gt) and np.all(boxes[j,2:] > boxes[j,:2]):\n            overlap = iou_one(boxes[j], gt)\n            overlap[used] = -1\n            g = int(overlap.argmax())\n            if overlap[g] >= .5:\n                used[g], assignment[j] = True, g\n    return used, assignment\n\n\ndef evaluate(rows, predictions, threshold):\n    require({r[\'image_id\'] for r in rows} == set(predictions), \'평가 이미지 집합과 예측 집합이 다릅니다.\')\n    details = []\n    for r in rows:\n        b,s = validate_prediction(predictions[r[\'image_id\']])\n        gt = np.asarray(r[\'gt\'],dtype=float).reshape(-1,4)\n        hit, assigned = match(gt,b,s,threshold)\n        small = (gt[:,2:]-gt[:,:2]).prod(axis=1) < 32**2\n        details.append({\'image_id\':r[\'image_id\'], \'image\':r[\'image\'], \'tp\':int(hit.sum()),\n                        \'fp\':int((s >= threshold).sum()-hit.sum()), \'fn\':int(len(gt)-hit.sum()),\n                        \'gt\':len(gt), \'small_tp\':int((hit & small).sum()), \'small_gt\':int(small.sum()),\n                        \'zero_area_fp\':int(((s >= threshold) & np.any(b[:,2:] == b[:,:2], axis=1)).sum())})\n    frame = pd.DataFrame(details)\n    totals = {k:int(frame[k].sum()) for k in (\'tp\',\'fp\',\'fn\',\'gt\',\'small_tp\',\'small_gt\',\'zero_area_fp\')}\n    tp,fp,fn = totals[\'tp\'],totals[\'fp\'],totals[\'fn\']\n    require(totals[\'gt\'] > 0, \'포트홀 GT가 없어 Recall을 계산할 수 없습니다.\')\n    measured = {**totals, \'images\':len(rows), \'recall\':tp/(tp+fn), \'precision\':tp/(tp+fp) if tp+fp else 0.,\n                \'precision_defined\':bool(tp+fp), \'f1\':2*tp/(2*tp+fp+fn), \'fp_per_image\':fp/len(rows),\n                \'small_recall\':totals[\'small_tp\']/totals[\'small_gt\'] if totals[\'small_gt\'] else None}\n    return measured, frame\n\n\ndef coco_ap(rows, predictions, max_det):\n    # Official COCO evaluator. AP maxDet is explicitly 300 (or recorded setting), not default 100.\n    from pycocotools.coco import COCO\n    from pycocotools.cocoeval import COCOeval\n    ground = {\'info\':{}, \'licenses\':[], \'images\':[], \'categories\':[{\'id\':1,\'name\':\'pothole\'}], \'annotations\':[]}\n    detected = []\n    for i,r in enumerate(rows,1):\n        ground[\'images\'].append({\'id\':i,\'width\':r[\'width\'],\'height\':r[\'height\']})\n        for x1,y1,x2,y2 in r[\'gt\']:\n            ground[\'annotations\'].append({\'id\':len(ground[\'annotations\'])+1,\'image_id\':i,\'category_id\':1,\n                                           \'bbox\':[x1,y1,x2-x1,y2-y1], \'area\':(x2-x1)*(y2-y1),\'iscrowd\':0})\n        b,s = validate_prediction(predictions[r[\'image_id\']])\n        for box,score in zip(b,s):\n            x1,y1,x2,y2 = map(float,box)\n            detected.append({\'image_id\':i,\'category_id\':1,\'bbox\':[x1,y1,x2-x1,y2-y1], \'score\':float(score)})\n    if not detected: return {\'ap50\':0., \'ap50_95\':0., \'ap_max_det\':max_det}\n    with contextlib.redirect_stdout(io.StringIO()):\n        gt = COCO(); gt.dataset=ground; gt.createIndex()\n        dt = gt.loadRes(detected)\n        ev = COCOeval(gt,dt,\'bbox\'); ev.params.maxDets=[1,10,int(max_det)]\n        ev.evaluate(); ev.accumulate()\n    precision = ev.eval[\'precision\'][:,:,0,0,2]\n    valid = precision[precision >= 0]\n    p50 = precision[np.isclose(ev.params.iouThrs, .5)]\n    return {\'ap50\':float(p50[p50>=0].mean()), \'ap50_95\':float(valid.mean()), \'ap_max_det\':max_det}\n\n\ndef infer_one(rows, record, settings, env, cfg):\n    import torch\n    from ultralytics import YOLO, RTDETR\n    signature = {\'version\':VERSION, \'code_sha\':file_sha(__file__), \'weight\':record[\'weight_sha256\'],\n                 \'model\':record[\'model\'], \'settings\':settings, \'environment\':env,\n                 \'images\':[{k:r[k] for k in (\'image_id\',\'width\',\'height\')} for r in rows]}\n    cache = Path(cfg[\'output_root\'])/\'prediction_cache\'/f\'{digest(signature)}.json\'\n    if cache.is_file():\n        obj = read_json(cache)\n        require(obj[\'signature\'] == signature, \'캐시 설정 불일치\')\n        pred = obj[\'predictions\']\n        require(set(pred) == {r[\'image_id\'] for r in rows}, \'캐시 이미지 누락\')\n        for p in pred.values(): validate_prediction(p)\n        print(f\'캐시 사용: fold{record["fold"]}/{record["model"]}\')\n        return pred,str(cache)\n    model = None\n    try:\n        model = (YOLO if record[\'model\'] == \'yolo\' else RTDETR)(record[\'weights\'])\n        require(len(model.names) == 1, f\'기존 포트홀 단일 클래스 가중치가 아닙니다: {model.names}\')\n        pred = {}\n        for n,r in enumerate(rows,1):\n            rgb = rgb_image(r[\'image\'])\n            require(pixel_sha(rgb) == r[\'image_id\'], f\'검사 후 이미지가 변경됐습니다: {r["image"]}\')\n            res = model.predict(source=np.ascontiguousarray(rgb[:,:,::-1]), device=env[\'device\'], verbose=False,\n                                save=False, **settings)[0]\n            require(res.boxes is not None, \'detect bbox 결과가 없습니다.\')\n            require(np.all(res.boxes.cls.cpu().numpy() == 0), \'예측 클래스가 0이 아닙니다.\')\n            p = {\'boxes\':res.boxes.xyxy.cpu().numpy().astype(float).tolist(),\n                 \'scores\':res.boxes.conf.cpu().numpy().astype(float).tolist()}\n            validate_prediction(p)\n            order = np.argsort(-np.asarray(p[\'scores\']),kind=\'stable\')[:int(settings[\'max_det\'])]\n            pred[r[\'image_id\']] = {k:[v[int(j)] for j in order] for k,v in p.items()}\n            if n % 20 == 0 or n == len(rows): print(f\'  {record["model"]}: {n}/{len(rows)}\')\n        save_json(cache, {\'signature\':signature,\'predictions\':pred})\n        return pred,str(cache)\n    finally:\n        del model\n        gc.collect()\n        if torch.cuda.is_available(): torch.cuda.empty_cache()\n\n\ndef summarize(frame):\n    output=[]\n    for (model,target),group in frame.groupby([\'model\',\'target_fp_per_image\']):\n        row={\'model\':model, \'source_val_fp_target\':float(target), \'checkpoints\':len(group),\n             \'unique_external_images\':int(group[\'images\'].iloc[0]), \'unique_external_gt\':int(group[\'gt\'].iloc[0])}\n        for metric in (\'recall\',\'precision\',\'f1\',\'fp_per_image\',\'small_recall\',\'ap50\',\'ap50_95\'):\n            if metric not in group: continue\n            values=pd.to_numeric(group[metric],errors=\'coerce\').dropna()\n            row[metric+\'_mean\']=float(values.mean()) if len(values) else None\n            row[metric+\'_sd\']=float(values.std(ddof=1)) if len(values)>1 else None\n        output.append(row)\n    return pd.DataFrame(output)\n\n\ndef paired_table(frame):\n    rows=[]\n    for (fold,target),g in frame.groupby([\'fold\',\'target_fp_per_image\']):\n        d=g.set_index(\'model\'); require(set(d.index)==set(MODELS), \'모델 pair가 없습니다.\')\n        y,r=d.loc[\'yolo\'],d.loc[\'rtdetr\']\n        rows.append({\'fold\':int(fold),\'source_val_fp_target\':float(target),\n                     \'recall_delta_pp_rtdetr_minus_yolo\':100*(r.recall-y.recall),\n                     \'small_recall_delta_pp_rtdetr_minus_yolo\':100*(float(r.small_recall)-float(y.small_recall)) if pd.notna(r.small_recall) and pd.notna(y.small_recall) else None,\n                     \'fp_per_image_delta_rtdetr_minus_yolo\':r.fp_per_image-y.fp_per_image,\n                     \'ap50_delta_pp_rtdetr_minus_yolo\':100*(r.ap50-y.ap50)})\n    return pd.DataFrame(rows)\n\n\ndef plots(summary, out):\n    import matplotlib.pyplot as plt\n    targets=sorted(summary.source_val_fp_target.unique())\n    fig,axs=plt.subplots(1,3,figsize=(14,4))\n    for i,model in enumerate(MODELS):\n        g=summary[summary.model==model].set_index(\'source_val_fp_target\').loc[targets]\n        for ax,key,scale,title in zip(axs,[\'recall\',\'small_recall\',\'fp_per_image\'],[100,100,1],[\'Recall (%)\',\'Small recall (%)\',\'Actual FP / image\']):\n            values=pd.to_numeric(g[key+\'_mean\'],errors=\'coerce\').to_numpy(dtype=float)*scale\n            err=pd.to_numeric(g[key+\'_sd\'],errors=\'coerce\').fillna(0).to_numpy(dtype=float)*scale\n            ax.bar(np.arange(len(targets))+(i-.5)*.36,values,.36,yerr=err,label=model,capsize=3)\n            ax.set_xticks(np.arange(len(targets)),[str(t) for t in targets]); ax.set_xlabel(\'Source validation FP target\')\n            ax.set_title(title); ax.grid(axis=\'y\',alpha=.2)\n    axs[0].legend()\n    fig.suptitle(\'External test: frozen weights and thresholds | mean / checkpoint SD (not confidence interval)\')\n    fig.tight_layout(); fig.savefig(Path(out)/\'external_comparison.png\',dpi=160); plt.close(fig)\n\n\ndef compare_source(frame, source, out):\n    path=Path(source[\'root\'])/\'outer_grid_by_fold.csv\'\n    if not path.is_file():\n        print(\'기존 outer_grid_by_fold.csv가 없어 기존/새 데이터 수치 비교표는 생략합니다.\')\n        return None\n    old=pd.read_csv(path)\n    keys=[\'fold\',\'model\',\'imgsz\',\'target_fp_per_image\']\n    values=[\'recall\',\'precision\',\'f1\',\'fp_per_image\',\'small_recall\',\'images\',\'gt\']\n    require(set(keys+values)<=set(old.columns), \'기존 outer_grid_by_fold.csv의 열 구조가 다릅니다.\')\n    old=old[keys+values].rename(columns={k:k+\'_source\' for k in values})\n    merged=frame.merge(old,on=keys,how=\'left\',validate=\'one_to_one\')\n    require(merged[\'recall_source\'].notna().all(), \'동일 fold/model/imgsz/목표의 기존 test 결과가 누락되었습니다.\')\n    for key in (\'recall\',\'precision\',\'f1\',\'small_recall\'):\n        merged[key+\'_delta_pp_external_minus_source\']=100*(pd.to_numeric(merged[key],errors=\'coerce\')-pd.to_numeric(merged[key+\'_source\'],errors=\'coerce\'))\n    merged[\'fp_per_image_delta_external_minus_source\']=merged.fp_per_image-merged.fp_per_image_source\n    merged.to_csv(Path(out)/\'source_vs_external_by_checkpoint.csv\',index=False,encoding=\'utf-8-sig\')\n    metrics=[k for k in merged if k.endswith(\'_source\') or \'_delta_\' in k]\n    metrics=[k for k in metrics if k not in (\'images_source\',\'gt_source\')]+[\'recall\',\'precision\',\'f1\',\'small_recall\',\'fp_per_image\']\n    for col in metrics: merged[col]=pd.to_numeric(merged[col],errors=\'coerce\')\n    aggregate=merged.groupby([\'model\',\'target_fp_per_image\'])[metrics].mean().reset_index()\n    aggregate.to_csv(Path(out)/\'source_vs_external_mean.csv\',index=False,encoding=\'utf-8-sig\')\n    print(\'기존/신규 비교표는 양쪽 모두 fold 체크포인트 평균입니다. 기존 보고서의 pooled OOF와 평균 방식이 다를 수 있습니다.\')\n    return aggregate\n\n\ndef comparison_cases(rows, predictions, records, out, count=6):\n    import matplotlib.pyplot as plt\n    target=max(r[\'target_fp_per_image\'] for r in records)\n    chosen={r[\'model\']:r for r in records if r[\'target_fp_per_image\']==target}\n    sample=np.linspace(0,len(rows)-1,min(count,len(rows)),dtype=int)\n    folder=Path(out)/\'cases\'; folder.mkdir(exist_ok=True)\n    paths=[]\n    for index in sample:\n        r=rows[index]; rgb=rgb_image(r[\'image\'])\n        fig,axs=plt.subplots(1,3,figsize=(18,5))\n        for ax in axs: ax.imshow(rgb); ax.axis(\'off\')\n        for x1,y1,x2,y2 in r[\'gt\']:\n            axs[0].add_patch(plt.Rectangle((x1,y1),x2-x1,y2-y1,fill=False,color=\'lime\',lw=2))\n        axs[0].set_title(f\'Ground truth: {len(r["gt"])}\')\n        for ax,model in zip(axs[1:],MODELS):\n            b,s=validate_prediction(predictions[model][r[\'image_id\']]); t=chosen[model][\'threshold\']\n            hit,assignment=match(r[\'gt\'],b,s,t)\n            for j in np.flatnonzero(s>=t):\n                x1,y1,x2,y2=b[j]\n                ax.add_patch(plt.Rectangle((x1,y1),x2-x1,y2-y1,fill=False,color=\'lime\' if assignment[j]>=0 else \'red\',lw=2))\n                ax.text(x1,y1,f\'{s[j]:.2f}\',color=\'yellow\',fontsize=8)\n            for g in np.flatnonzero(~hit):\n                x1,y1,x2,y2=r[\'gt\'][g]\n                ax.add_patch(plt.Rectangle((x1,y1),x2-x1,y2-y1,fill=False,color=\'orange\',lw=2,ls=\'--\'))\n            ax.set_title(f\'{model}: TP={hit.sum()}, FP={(s>=t).sum()-hit.sum()}, FN={len(hit)-hit.sum()}\')\n        fig.suptitle(f\'Predefined sample {index+1} | green TP, red FP, orange dashed FN\')\n        fig.tight_layout(); p=folder/f\'case_{index+1:05d}.png\'; fig.savefig(p,dpi=130); plt.close(fig); paths.append(str(p))\n    return paths\n\n\ndef run(cfg, prepared, source, out):\n    import torch\n    import ultralytics\n    import importlib.metadata\n    # Fail before expensive inference if the evaluator dependency is unavailable.\n    import pycocotools.coco\n    require(prepared[\'signature\']==digest({\'rows\':prepared[\'rows\'],\'config\':dataset_settings(cfg)}), \'준비 후 데이터 설정 변경. prepare부터 다시 실행하세요.\')\n    require(cfg.get(\'labels_reviewed\') is True, \'정답 박스 미리보기와 클래스 기준을 확인한 뒤 labels_reviewed=True로 설정하세요.\')\n    require(cfg.get(\'external_independence_checked\') is True, \'기존 학습/검증 이미지와 중복하지 않는 새 데이터인지 확인 후 external_independence_checked=True로 설정하세요.\')\n    require(ultralytics.__version__ == cfg[\'required_ultralytics\'], f\'Ultralytics={ultralytics.__version__}; 설치 셀 실행 후 런타임 재시작 필요\')\n    require(source[\'protocol\'][\'environment\'][\'ultralytics\']==ultralytics.__version__, \'7주차와 Ultralytics 버전이 다릅니다.\')\n    require(int(cfg[\'imgsz\'])==source[\'inference\'][\'imgsz\'], \'모델/threshold 준비 후 imgsz 변경\')\n    require(set(map(float,cfg[\'targets\']))=={r[\'target_fp_per_image\'] for r in source[\'records\']}, \'threshold 준비 후 targets 변경\')\n    require(set(map(int,cfg[\'folds\']))=={r[\'fold\'] for r in source[\'records\']}, \'threshold 준비 후 folds 변경\')\n    device=cfg.get(\'device\',\'auto\')\n    if device==\'auto\': device=0 if torch.cuda.is_available() else \'cpu\'\n    require(str(device)==\'cpu\' or torch.cuda.is_available(), \'GPU가 없습니다. 런타임 유형을 GPU로 바꾸세요.\')\n    torch.manual_seed(43)\n    if torch.cuda.is_available(): torch.cuda.manual_seed_all(43)\n    torch.backends.cudnn.benchmark=False\n    torch.backends.cudnn.deterministic=True\n    if hasattr(torch.backends,\'cuda\'): torch.backends.cuda.matmul.allow_tf32=False\n    torch.backends.cudnn.allow_tf32=False\n    env={\'ultralytics\':ultralytics.__version__,\'torch\':torch.__version__,\'cuda\':torch.version.cuda,\n         \'device\':device,\'gpu\':torch.cuda.get_device_name(int(device)) if str(device)!=\'cpu\' else None,\n         \'numpy\':np.__version__,\'pycocotools\':importlib.metadata.version(\'pycocotools\')}\n    rows=prepared[\'rows\']; out=Path(out)\n    require(not (out/\'completed.json\').exists(), \'이미 완료된 출력 폴더입니다. 새 실행은 셀 4부터 새 폴더를 만드세요.\')\n    for path,sha in source[\'source_hashes\'].items(): require(file_sha(path)==sha, \'7주차 기록이 변경됐습니다.\')\n    for r in rows:\n        require(file_sha(r[\'image\'])==r[\'image_sha256\'] and file_sha(r[\'json\'])==r[\'label_sha256\'], f\'검사 후 원본이 변경됐습니다: {r["image"]}\')\n    for r in { (r[\'fold\'],r[\'model\']):r for r in source[\'records\']}.values():\n        require(file_sha(r[\'weights\'])==r[\'weight_sha256\'], \'모델 준비 후 가중치 변경\')\n    protocol={\'version\':VERSION,\'config\':cfg,\'dataset\':prepared[\'audit\'],\'dataset_signature\':prepared[\'signature\'],\n              \'environment\':env,\'source_environment\':source[\'protocol\'][\'environment\'], \'inference\':source[\'inference\'],\n              \'source_hashes\':source[\'source_hashes\'],\'operating_points\':source[\'records\'], \'code_sha256\':file_sha(__file__),\n              \'limits\':[\'No training, calibration, or model selection on external test.\',\n                        \'Actual external FP budgets are not guaranteed.\', \'Architecture-native preprocessing; RT-DETR has no extra NMS.\',\n                        \'COCO AP uses all cached candidates at the recorded confidence floor and maxDet.\',\n                        \'Same external images reused across fold checkpoints; do not sum unique sample counts.\',\n                        \'Checkpoint SD is not an independent-sample confidence interval.\', \'Small uses original-pixel area < 32^2; image resolution affects this stratum.\',\n                        \'Positive-only data do not estimate general-road false alarm prevalence.\',\n                        \'Finite zero-area predictions count as FP; inverted/nonfinite predictions fail.\']}\n    save_json(out/\'external_protocol.json\',protocol)\n    shutil.copyfile(__file__,out/\'external_pothole_eval.py\')\n    records=[]; images=[]; caches=[]; case_paths=[]\n    first_fold=min(r[\'fold\'] for r in source[\'records\'])\n    for fold in sorted({r[\'fold\'] for r in source[\'records\']}):\n        sample_predictions={}\n        for model in MODELS:\n            ops=[r for r in source[\'records\'] if r[\'fold\']==fold and r[\'model\']==model]\n            print(f\'Fold {fold}: {model}, imgsz={cfg["imgsz"]}, external images={len(rows)}\')\n            pred,cache=infer_one(rows,ops[0],source[\'inference\'],env,cfg)\n            ap=coco_ap(rows,pred,int(source[\'inference\'][\'max_det\']))\n            caches.append({\'fold\':fold,\'model\':model,\'cache\':cache})\n            for op in ops:\n                metric,detail=evaluate(rows,pred,op[\'threshold\'])\n                meta={k:op[k] for k in (\'fold\',\'model\',\'imgsz\',\'target_fp_per_image\',\'threshold\')}\n                records.append({**meta,**metric,**ap})\n                for k,v in meta.items(): detail[k]=v\n                images.append(detail)\n            if fold==first_fold: sample_predictions[model]=pred\n        pd.DataFrame(records).to_csv(out/\'metrics_by_checkpoint.partial.csv\',index=False,encoding=\'utf-8-sig\')\n        if fold==first_fold:\n            case_paths=comparison_cases(rows,sample_predictions,[r for r in source[\'records\'] if r[\'fold\']==fold],out,int(cfg.get(\'case_images\',6)))\n    frame=pd.DataFrame(records); summary=summarize(frame); paired=paired_table(frame)\n    frame.to_csv(out/\'metrics_by_checkpoint.csv\',index=False,encoding=\'utf-8-sig\')\n    summary.to_csv(out/\'model_summary.csv\',index=False,encoding=\'utf-8-sig\')\n    paired.to_csv(out/\'paired_model_differences.csv\',index=False,encoding=\'utf-8-sig\')\n    pd.concat(images,ignore_index=True).to_csv(out/\'per_image_metrics.csv\',index=False,encoding=\'utf-8-sig\')\n    save_json(out/\'prediction_cache_index.json\',caches)\n    plots(summary,out)\n    source_comparison=compare_source(frame,source,out)\n    save_json(out/\'completed.json\',{\'status\':\'complete\',\'protocol_sha256\':digest(protocol),\'unique_images\':len(rows),\n                                    \'checkpoints\':len(caches),\'metric_rows\':len(frame),\'cases\':case_paths})\n    print(f\'완료: {out}\')\n    return {\'out\':out,\'metrics\':frame,\'summary\':summary,\'paired\':paired,\'source_comparison\':source_comparison,\'cases\':case_paths}\n'
engine_path = Path('/content/external_pothole_eval.py')
engine_path.write_text(ENGINE_SOURCE, encoding='utf-8')
spec = importlib.util.spec_from_file_location('external_pothole_eval', engine_path)
external = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = external
spec.loader.exec_module(external)
_rows = [{'image_id':'ap_smoke','width':100,'height':80,'gt':[[10,10,30,30]]}]
_pred = {'ap_smoke': {'boxes':[[10,10,30,30]],'scores':[.9]}}
_ap = external.coco_ap(_rows, _pred, 300)
assert abs(_ap['ap50_95'] - 1.) < 1e-9
assert external.coco_ap(_rows, {'ap_smoke':{'boxes':[],'scores':[]}}, 300)['ap50']==0.
print('평가 엔진', external.VERSION, '| AP smoke check OK')


## 셀 4 — 폴더 경로 설정
`data_dir`를 **이미지와 JSON이 함께 들어 있는 실제 폴더**로 바꾸세요. 아래 7주차 경로는 이전 대화의 실행 경로를 넣었습니다. 실제 위치가 다르면 수정하세요. 출력은 새 실행 폴더에 저장됩니다.

포트홀 외 클래스는 포트홀 GT로 사용하지 않습니다. 실제 포트홀 이름이 다르면 `target_names`를 수정하세요. 숫자 클래스 ID만 있고 JSON 안에 클래스표가 없다면 확인된 매핑을 `category_id_to_name`에 입력해야 합니다.


In [ ]:
CFG = {
    'data_dir': '/content/drive/MyDrive/Pothole2/',  # ★ 실제 이미지+JSON 폴더
    'week7_out': '/content/drive/MyDrive/Pothole/week7_fp_resolution_results/fp_resolution_20261031_113834_664819_UTC',
    'output_root': '/content/drive/MyDrive/Pothole2/korean_external_results',
    'recursive': True,
    'target_names': ['포트홀', 'pothole', 'potholes'],
    'category_id_to_name': {},  # 예: {'26':'포트홀'} — 실제 클래스표 확인 전 입력 금지
    'bbox_format': None,       # 셀 5의 미리보기를 확인한 후 셀 6에서 지정
    'bbox_units': 'pixels',    # 0~1 정규화 좌표라면 'normalized'
    'imgsz': 800,
    'folds': [1,2,3,4,5],
    'targets': [0.2, 0.5],     # 원 데이터 inner_val에서 정했던 FP/image 목표
    'device': 'auto',         # GPU가 있으면 0, 없으면 CPU. GPU 필수로 쓰려면 0
    'required_ultralytics': '8.4.183',
    'path_remap': {},         # 예: {'/예전/폴더':'/현재/폴더'}
    'weight_overrides': {},   # 예: {1:{'yolo':'/.../best.pt','rtdetr':'/.../best.pt'}}
    'labels_reviewed': False,
    'external_independence_checked': False,
    'case_images': 6,
}
from datetime import datetime, timezone
OUT = Path(CFG['output_root']) / datetime.now(timezone.utc).strftime('external_%Y%m%d_%H%M%S_%f_UTC')
OUT.mkdir(parents=True, exist_ok=False)
print('이번 실행 폴더:', OUT)


## 셀 5 — 이미지·JSON 연결과 좌표 해석 미리보기
누락 JSON을 빈 라벨로 처리하지 않습니다. 모든 이미지에 같은 이름의 JSON이 연결되어야 합니다. `bbox` 숫자 4개의 순서는 이름만으로 알 수 없으므로 두 후보를 나란히 표시합니다. **범위 안에 들어가는지만으로 올바른 형식을 판단하지 말고, 제공 설명서와 박스 위치를 함께 확인하세요.**

첫 이미지들이 포트홀을 포함하지 않으면 아래 `head(3)`을 다른 이미지 구간으로 바꿔 확인할 수 있습니다. 지원하지 않는 JSON이면 구조 출력이 남으므로 원 JSON 한 개와 함께 공유하세요.


In [ ]:
PAIRS, ORPHAN_JSON = external.inventory(CFG)
display(PAIRS.head(10))
SAMPLES = external.inspect_samples(PAIRS, CFG, OUT)
external.format_preview(PAIRS.head(3), CFG, OUT)


## 셀 6 — 확인한 좌표 형식 지정

| JSON bbox의 뜻 | 설정 |
|---|---|
| `[왼쪽 x, 위쪽 y, 오른쪽 x, 아래쪽 y]` | `'xyxy'` |
| `[왼쪽 x, 위쪽 y, 너비, 높이]` | `'xywh'` |
| `[중심 x, 중심 y, 너비, 높이]` | `'cxcywh'` |

AI Hub JSON이라고 해서 모두 COCO의 xywh는 아닙니다. 실제 내용을 확인하세요. bbox가 `xmin/ymin/xmax/ymax` 같은 명시적 dict이거나 `[[x1,y1],[x2,y2]]`이면 자동 해석되므로 None을 유지해도 됩니다. 이 경우에도 다음 셀의 정답 위치를 검수하세요.


In [ ]:
# 아래 None을 확인한 'xyxy' 또는 'xywh' 등으로 바꾸세요.
# 숫자 4개 bbox인데 None이면 다음 셀이 설명과 함께 멈춥니다.
CFG['bbox_format'] = None
CFG['bbox_units'] = 'pixels'
print('좌표 해석:', CFG['bbox_format'], '| 단위:', CFG['bbox_units'])


## 셀 7 — 전체 라벨 검사와 정답 미리보기
이미지 크기·내부 파일명·이미지 ID·클래스·좌표·동일 pixel 중복을 검사합니다. 원본 좌표를 임의로 잘라내거나 문제가 있는 이미지를 조용히 제외하지 않습니다. 정상 완료 시 이미지 수, 양성/음성 이미지 수, 포트홀 수, 작은 포트홀 수를 출력합니다.

숫자 0~1이면 units를 확인하세요. 면적 0인 GT는 오류입니다. 예측에서 생기는 유한한 면적 0 박스는 나중에 FP로 유지합니다. 파일명에서 만든 group_hint는 실제 촬영 그룹이 확인된 값이 아닙니다.


In [ ]:
PREPARED = external.prepare(CFG, OUT)
external.preview_ground_truth(PREPARED, OUT)


## 셀 8 — 기존 가중치와 threshold 연결
7주차의 baseline 입력 800에 해당하는 inner validation threshold를 불러옵니다. 완료 상태와 결정 파일 해시를 확인하고, 각 best.pt의 해시가 기존 실험과 같은지도 확인합니다. **새 데이터의 성능을 보고 fold나 threshold를 고르지 않습니다.**

이 셀에서 경로 오류가 나면 원 가중치 위치를 셀 4의 `path_remap`/`weight_overrides`에 입력한 후 다시 실행하세요. `decisions_locked.json`이 없으면 0.25 같은 임의값으로 대체하지 않습니다.


In [ ]:
SOURCE = external.load_week7(CFG)
display(pd.DataFrame(SOURCE['records'])[['fold','model','imgsz','target_fp_per_image','threshold','weights']])
print('추론 설정:', SOURCE['inference'])


## 셀 9 — 검수 확인 후 외부 평가 실행
다음 두 확인값은 실제로 확인한 경우에만 True로 바꾸세요.

- 정답 박스가 포트홀을 감싸고, 라벨 누락·클래스 오해가 없는가? 맨홀·보수 흔적·크랙은 포트홀과 구분했는가?
- 새 이미지가 기존 학습·validation·test 이미지와 중복되지 않는가? 인접 프레임만 과도하게 반복 선정하지 않았는가?

GPU 자동 사용, batch=1, FP32, 기존 NMS/전처리와 confidence floor를 유지합니다. RT-DETR에는 추가 NMS를 하지 않습니다. 실행이 중단되면 설정과 원본이 같을 때 완료된 모델 캐시를 재사용할 수 있습니다. 환경은 모두 기록되며 CUDA/torch 차이가 있으면 기존 실행과의 수치 비교 시 함께 보고하세요.


In [ ]:
CFG['labels_reviewed'] = False                  # 정답 박스를 확인했으면 True
CFG['external_independence_checked'] = False    # 기존 데이터와 독립이면 True

RESULT = external.run(CFG, PREPARED, SOURCE, OUT)
print('완료 폴더:', RESULT['out'])


## 셀 10 — 두 모델과 기존/새 데이터 비교

Recall/Precision/F1/AP/small Recall의 CSV 값은 0~1 비율입니다. 그래프의 Recall은 %입니다. `*_delta_pp_*`는 퍼센트포인트 차이입니다.

- `model_summary.csv`: 같은 외부 세트에 대한 5개 체크포인트 평균과 sample SD. SD는 신뢰구간이 아닙니다.
- `paired_model_differences.csv`: 같은 fold·목표에서 RT-DETR − YOLO 차이.
- `source_vs_external_mean.csv`: 기존/새 데이터 모두 fold 평균으로 비교. 기존 보고서의 pooled OOF 수치와 평균 방식이 다를 수 있습니다.
- `metrics_by_checkpoint.csv`: TP/FP/FN, 실제 FP/image, 작은 포트홀 Recall, AP50/AP50–95 등 상세 결과.
- `per_image_metrics.csv`, `cases/`: 이미지별 결과와 사전에 정한 간격으로 뽑은 사례.

AP는 COCO evaluator로 전체 저장 후보를 평가하며, 운영 threshold별 Recall과 별개입니다. 기본 최대 검출 수는 원 기록의 300이므로 기본 COCO maxDet=100 결과와 혼동하지 마세요. 낮은 conf_floor 아래 예측은 관측하지 못합니다. 작은 객체는 원본 면적 32² 미만이며 촬영 해상도가 다른 데이터 사이에서 동일한 물리 크기를 뜻하지 않습니다.

새 test의 FP/image가 0.2/0.5를 넘을 수 있습니다. 동일한 원 validation 목표를 썼다는 뜻이지 실제 오탐이 같다는 뜻은 아닙니다. 같은 외부 이미지가 fold마다 반복 사용되므로 전체 이미지 수를 5배로 세지 않습니다. 포트홀 중심으로 모은 세트는 실제 일반 주행의 유병률/오탐 부담을 대표하지 않습니다.


In [ ]:
display(RESULT['summary'])
display(RESULT['paired'])
if RESULT['source_comparison'] is not None:
    display(RESULT['source_comparison'])
display(DisplayImage(filename=str(OUT/'external_comparison.png')))
for path in RESULT['cases'][:2]:
    display(DisplayImage(filename=path))


## 분석할 때 전달할 파일
실행 후 `model_summary.csv`, `metrics_by_checkpoint.csv`, `source_vs_external_mean.csv`(생성된 경우), `dataset_audit.json`, `external_protocol.json`, `external_comparison.png`를 공유하면 새 데이터에서 성능과 우열이 유지되는지 분석할 수 있습니다. `completed.json`이 있어야 전체 실행 완료입니다. `.partial.csv`는 중간 저장입니다.

실제 JSON 구조가 지원 형태와 다르면 `json_schema_samples.json` 또는 원 JSON 1개를 먼저 공유하세요. 아직 이 노트북을 사용자의 실제 이미지/JSON/가중치로 실행하지 않았습니다. 로컬에서는 파일 연결, 좌표 변환, 누락 라벨, 중복, TP/FP/FN, 출처 검사 등을 합성 자료로 검증했습니다.

### 참고
- AI Hub dataset 572: https://aihub.or.kr/aihubdata/data/view.do?dataSetSn=572
- Ultralytics prediction API: https://docs.ultralytics.com/modes/predict/
- RT-DETR API: https://docs.ultralytics.com/models/rtdetr/
- COCO evaluator: https://github.com/cocodataset/cocoapi/tree/master/PythonAPI/pycocotools
